In [ ]:
# Experts:
# expert 0 -> fine-tuned on hard DAY images
# expert 1 -> fine-tuned on hard NIGHT images
# expert 2 -> fine-tuned on hard DAWN_DUSK images
# expert 3 -> fine-tuned on ALL hard images
#
# Gating:
# trained to route day->0, night->1, dawn_dusk->2
# inference uses Top-1 specialist + expert 3 (general expert)

import os
import csv
import json
import copy
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.ops import batched_nms, box_iou


# PATHS
BDD_DATA_ROOT = r"C:\Users\ao0952\Desktop\dataset\bdd100\archive\val"
BDD_IMAGES_DIR = os.path.join(BDD_DATA_ROOT, "images")
BDD_ANN_JSON   = os.path.join(BDD_DATA_ROOT, "annotations", "bdd100k_labels_images_val.json")

BDD_SPLITS = {
    "day": r"C:\Users\ao0952\Desktop\dataset\bdd100\archive\day.txt",
    "night": r"C:\Users\ao0952\Desktop\dataset\bdd100\archive\night.txt",
    "dawn_dusk": r"C:\Users\ao0952\Desktop\dataset\bdd100\archive\dawn_dusk.txt",
}

# CSV files created from previous hard-image analysis
HARD_CSVS = {
    "day": r"C:\Users\ao0952\Desktop\bdd_error_analysis\day\day_human_review.csv",
    "night": r"C:\Users\ao0952\Desktop\bdd_error_analysis\night\night_human_review.csv",
    "dawn_dusk": r"C:\Users\ao0952\Desktop\bdd_error_analysis\dawn_dusk\dawn_dusk_human_review.csv",
}

SAVE_DIR = r"C:\Users\ao0952\Desktop\bdd_hard_finetuned_moe"
os.makedirs(SAVE_DIR, exist_ok=True)

EXPERT_SAVE_PATH = os.path.join(SAVE_DIR, "bdd_hard_finetuned_expertsall3.pth")
GATING_SAVE_PATH = os.path.join(SAVE_DIR, "bdd_hard_gatingall3.pth")


# CONFIG
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_EXPERTS = 4
NUM_SPECIALISTS = 3

# COCO person id = 1
COCO_PERSON_LABEL = 1

# VOC person id = 15
VOC_PERSON_LABEL = 15

SCORE_THRESH = 0.05
NMS_IOU_THRESH = 0.5
MAX_DETS = 300

IOU_THRESHOLDS = [round(x / 100, 2) for x in range(50, 100, 5)]

BATCH_SIZE_EXPERT = 1
BATCH_SIZE_GATING = 4

NUM_EPOCHS_EXPERT = 50
NUM_EPOCHS_GATING = 40

LR_EXPERT = 1e-5
LR_GATING = 1e-4

TRAIN_BACKBONE = False

COCO_TO_VOC = {
    1: 15,  # person
}



# UTILS
def collate_fn_list(batch):
    return batch


def normalize_img_name_from_txt(line):
    line = line.strip()
    if not line:
        return None
    if line.lower() == "jpg":
        return None
    return os.path.basename(line)


def load_split_image_names(split_txt_path):
    image_names = []
    with open(split_txt_path, "r", encoding="utf-8") as f:
        for line in f:
            name = normalize_img_name_from_txt(line)
            if name is not None:
                image_names.append(name)
    return image_names


def load_hard_image_names_from_csv(csv_path):
    image_names = []

    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            img_name = row.get("image_name", None)
            if img_name is not None and img_name.strip():
                image_names.append(os.path.basename(img_name.strip()))

    return image_names


def load_bdd_annotations(annotation_json_path, target_image_names):
    with open(annotation_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    target_set = set(target_image_names)
    ann_dict = {}

    for item in data:
        img_name = item.get("name")

        if img_name not in target_set:
            continue

        boxes = []

        for ann in item.get("labels", []):
            if ann.get("category") != "person":
                continue

            box2d = ann.get("box2d", None)
            if box2d is None:
                continue

            x1 = float(box2d["x1"])
            y1 = float(box2d["y1"])
            x2 = float(box2d["x2"])
            y2 = float(box2d["y2"])

            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])

        ann_dict[img_name] = boxes

    for img_name in target_image_names:
        ann_dict.setdefault(img_name, [])

    return ann_dict


# DATASETS
class BDDPersonDataset(Dataset):
    def __init__(self, images_dir, ann_json_path, image_names, split_name=None):
        self.images_dir = images_dir
        self.image_names = list(image_names)
        self.split_name = split_name
        self.ann_dict = load_bdd_annotations(ann_json_path, self.image_names)

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.images_dir, img_name)

        img = Image.open(img_path).convert("RGB")
        boxes = self.ann_dict.get(img_name, [])

        target_eval = {
            "image_id": img_name,
            "boxes": boxes
        }

        return img, target_eval


class BDDHardTrainDataset(Dataset):
    def __init__(self, images_dir, ann_json_path, image_names, split_label):
        self.images_dir = images_dir
        self.image_names = list(image_names)
        self.split_label = split_label
        self.ann_dict = load_bdd_annotations(ann_json_path, self.image_names)

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.images_dir, img_name)

        img = Image.open(img_path).convert("RGB")
        boxes = self.ann_dict.get(img_name, [])

        boxes_t = torch.tensor(boxes, dtype=torch.float32)

        if len(boxes) > 0:
            labels_t = torch.ones((len(boxes),), dtype=torch.int64) * COCO_PERSON_LABEL
        else:
            labels_t = torch.zeros((0,), dtype=torch.int64)

        target_train = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([idx])
        }

        return img, target_train, self.split_label, img_name


class CombinedHardDataset(Dataset):
    def __init__(self, datasets):
        self.items = []

        for d in datasets:
            for i in range(len(d)):
                self.items.append((d, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        d, i = self.items[idx]
        return d[i]


# MODEL
def get_coco_faster_rcnn_trainable(train_backbone=False):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

    for p in model.parameters():
        p.requires_grad = False

    # train RPN + ROI heads
    for p in model.rpn.parameters():
        p.requires_grad = True

    for p in model.roi_heads.parameters():
        p.requires_grad = True

    if train_backbone:
        for p in model.backbone.parameters():
            p.requires_grad = True

    return model


class ImageGatingNetwork(nn.Module):
    def __init__(self, num_experts=3, hidden_dim=128):
        super().__init__()

        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
        resnet = torchvision.models.resnet18(weights=weights)

        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.fc1 = nn.Linear(512, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_experts)

    def forward(self, x):
        x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        x = TF.normalize(
            x,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        f = self.features(x).view(x.size(0), -1)
        h = torch.relu(self.fc1(f))

        return self.fc2(h)


def map_coco_output_to_voc(out):
    boxes = out["boxes"]
    scores = out["scores"]
    labels = out["labels"]

    new_boxes, new_scores, new_labels = [], [], []

    for b, s, l in zip(boxes, scores, labels):
        coco_id = int(l.item())

        if coco_id in COCO_TO_VOC:
            new_boxes.append(b)
            new_scores.append(s)
            new_labels.append(torch.tensor(COCO_TO_VOC[coco_id], device=l.device))

    if len(new_boxes) == 0:
        return {
            "boxes": torch.empty((0, 4), device=boxes.device),
            "scores": torch.empty((0,), device=boxes.device),
            "labels": torch.empty((0,), dtype=torch.long, device=boxes.device)
        }

    return {
        "boxes": torch.stack(new_boxes),
        "scores": torch.stack(new_scores),
        "labels": torch.stack(new_labels).long()
    }


# TRAIN EXPERTS
def train_one_expert(expert, loader, expert_name, num_epochs=3, lr=1e-5):
    expert.to(device)

    params = [p for p in expert.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)

    print(f"\nTraining {expert_name}")
    print("Trainable parameters:", sum(p.numel() for p in params))

    for epoch in range(num_epochs):
        expert.train()
        total_loss = 0.0

        pbar = tqdm(loader, desc=f"{expert_name} epoch {epoch+1}/{num_epochs}")

        for batch in pbar:
            images = []
            targets = []

            for img_pil, target, split_label, img_name in batch:
                if target["boxes"].numel() == 0:
                    continue

                img_t = TF.to_tensor(img_pil).to(device)

                target_t = {
                    "boxes": target["boxes"].to(device),
                    "labels": target["labels"].to(device),
                    "image_id": target["image_id"].to(device)
                }

                images.append(img_t)
                targets.append(target_t)

            if len(images) == 0:
                continue

            loss_dict = expert(images, targets)
            loss = sum(v for v in loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item())
            pbar.set_postfix(loss=float(loss.item()))

        avg_loss = total_loss / max(1, len(loader))
        print(f"{expert_name} epoch {epoch+1}: loss={avg_loss:.4f}")

    expert.eval()
    return expert


# TRAIN GATING
def train_gating_network(gating_net, loader, num_epochs=5, lr=1e-4):
    gating_net.to(device)
    gating_net.train()

    optimizer = torch.optim.AdamW(gating_net.parameters(), lr=lr, weight_decay=1e-4)

    ce_loss = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(loader, desc=f"Gating epoch {epoch+1}/{num_epochs}")

        for batch in pbar:
            images = []
            labels = []

            for img_pil, target, split_label, img_name in batch:
                img_t = TF.to_tensor(img_pil)
                images.append(img_t)
                labels.append(split_label)

            images = torch.stack(images, dim=0).to(device)
            labels = torch.tensor(labels, dtype=torch.long).to(device)

            logits = gating_net(images)
            loss = ce_loss(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item())

            preds = torch.argmax(logits, dim=1)
            correct += int((preds == labels).sum().item())
            total += labels.numel()

            acc = correct / max(1, total)
            pbar.set_postfix(loss=float(loss.item()), acc=float(acc))

        print(f"Gating epoch {epoch+1}: loss={total_loss/max(1,len(loader)):.4f}, acc={correct/max(1,total):.4f}")

    gating_net.eval()
    return gating_net


# INFERENCE
@torch.no_grad()
def inference_moe(
    gating_net,
    experts,
    image_pil,
    k_top=1,
    score_thresh=0.05,
    iou_thresh=0.5,
    max_dets=300
):
    gating_net.eval()

    for e in experts:
        e.eval()

    img = TF.to_tensor(image_pil).to(device)
    img_b = img.unsqueeze(0)

    # Gating is trained only for the three specialist experts:
    # 0 -> day, 1 -> night, 2 -> dawn_dusk
    gate_logits = gating_net(img_b)
    gate_probs = F.softmax(gate_logits, dim=1)





    # Select Top-1 specialist
    specialist_idx = int(torch.argmax(gate_probs, dim=1).item())

    # Gating confidence of the selected specialist
    specialist_weight = gate_probs[0, specialist_idx]

    # General expert always keeps full confidence
    general_weight = torch.tensor(
        1.0,
        dtype=torch.float32,
        device=device
    )

    selected_expert_indices = [specialist_idx, 3]

    weights = torch.stack([
        specialist_weight,
        general_weight
    ])









    all_boxes, all_scores, all_labels = [], [], []

    for j, e_idx in enumerate(selected_expert_indices):
        expert = experts[e_idx]

        out = expert([img])[0]
        out = map_coco_output_to_voc(out)

        boxes = out["boxes"]
        scores = out["scores"]
        labels = out["labels"]

        keep = scores >= score_thresh

        boxes = boxes[keep]
        scores = scores[keep]
        labels = labels[keep]

        if boxes.numel() == 0:
            continue

        scores = scores * weights[j]

        all_boxes.append(boxes)
        all_scores.append(scores)
        all_labels.append(labels)

    if len(all_boxes) == 0:
        return {
            "boxes": torch.empty((0, 4)),
            "scores": torch.empty((0,)),
            "labels": torch.empty((0,), dtype=torch.long)
        }

    boxes = torch.cat(all_boxes, dim=0)
    scores = torch.cat(all_scores, dim=0)
    labels = torch.cat(all_labels, dim=0)

    keep = batched_nms(boxes, scores, labels, iou_thresh)
    keep = keep[:max_dets]

    return {
        "boxes": boxes[keep].detach().cpu(),
        "scores": scores[keep].detach().cpu(),
        "labels": labels[keep].detach().cpu()
    }


# AP
def compute_ap_from_pr(precisions, recalls):
    if len(precisions) == 0:
        return 0.0

    mrec = [0.0] + list(recalls) + [1.0]
    mpre = [0.0] + list(precisions) + [0.0]

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    ap = 0.0

    for i in range(1, len(mrec)):
        ap += (mrec[i] - mrec[i - 1]) * mpre[i]

    return ap


def compute_ap_at_iou(predictions_all, gt_dict, iou_thresh):
    total_gts = sum(len(v) for v in gt_dict.values())

    if total_gts == 0:
        return 0.0, 0.0, 0.0, 0.0

    predictions_all = sorted(predictions_all, key=lambda x: x["score"], reverse=True)

    gt_records = {}

    for img_name, boxes in gt_dict.items():
        gt_records[img_name] = {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if len(boxes) > 0 else torch.zeros((0, 4)),
            "matched": [False] * len(boxes)
        }

    tp, fp = [], []

    for pred in predictions_all:
        img_name = pred["img_name"]
        pred_box = torch.tensor(pred["box"], dtype=torch.float32).unsqueeze(0)

        gt_boxes = gt_records[img_name]["boxes"]
        matched = gt_records[img_name]["matched"]

        if gt_boxes.numel() == 0:
            tp.append(0)
            fp.append(1)
            continue

        ious = box_iou(pred_box, gt_boxes).squeeze(0)
        best_iou, best_idx = torch.max(ious, dim=0)

        if best_iou.item() >= iou_thresh and not matched[best_idx.item()]:
            tp.append(1)
            fp.append(0)
            matched[best_idx.item()] = True
        else:
            tp.append(0)
            fp.append(1)

    if len(tp) == 0:
        return 0.0, 0.0, 0.0, 0.0

    tp_cum = torch.tensor(tp).cumsum(0).float()
    fp_cum = torch.tensor(fp).cumsum(0).float()

    precisions = tp_cum / (tp_cum + fp_cum + 1e-8)
    recalls = tp_cum / (total_gts + 1e-8)

    ap = compute_ap_from_pr(precisions.tolist(), recalls.tolist())

    final_tp = tp_cum[-1].item()
    final_fp = fp_cum[-1].item()
    final_fn = total_gts - final_tp

    precision = final_tp / (final_tp + final_fp + 1e-8)
    recall = final_tp / (final_tp + final_fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return float(ap), float(precision), float(recall), float(f1)


@torch.no_grad()
def evaluate_bdd_person(gating_net, experts, data_loader, split_name):
    gt_dict = {}
    predictions_all = []

    pbar = tqdm(total=len(data_loader.dataset), desc=f"BDD Person Eval [{split_name}]")

    for batch in data_loader:
        for img, target in batch:
            img_name = target["image_id"]
            gt_boxes = target["boxes"]

            gt_dict[img_name] = gt_boxes

            det = inference_moe(
                gating_net=gating_net,
                experts=experts,
                image_pil=img,
                k_top=1,
                score_thresh=SCORE_THRESH,
                iou_thresh=NMS_IOU_THRESH,
                max_dets=MAX_DETS
            )

            boxes = det["boxes"]
            scores = det["scores"]
            labels = det["labels"]

            for i in range(len(boxes)):
                if int(labels[i]) == VOC_PERSON_LABEL and float(scores[i]) >= SCORE_THRESH:
                    predictions_all.append({
                        "img_name": img_name,
                        "score": float(scores[i]),
                        "box": [float(x) for x in boxes[i].tolist()]
                    })

            pbar.update(1)

    pbar.close()

    total_gts = sum(len(v) for v in gt_dict.values())

    ap_by_iou = {}
    precision_by_iou = {}
    recall_by_iou = {}
    f1_by_iou = {}

    for iou_t in IOU_THRESHOLDS:
        ap, precision, recall, f1 = compute_ap_at_iou(
            predictions_all=predictions_all,
            gt_dict=gt_dict,
            iou_thresh=iou_t
        )

        ap_by_iou[iou_t] = ap
        precision_by_iou[iou_t] = precision
        recall_by_iou[iou_t] = recall
        f1_by_iou[iou_t] = f1

    ap50 = ap_by_iou[0.50]
    ap75 = ap_by_iou[0.75]
    map_50_95 = sum(ap_by_iou.values()) / len(ap_by_iou)

    precision_50 = precision_by_iou[0.50]
    recall_50 = recall_by_iou[0.50]
    f1_50 = f1_by_iou[0.50]

    return {
        "num_images": len(gt_dict),
        "num_gt": int(total_gts),
        "num_pred": int(len(predictions_all)),

        "precision": float(precision_50),
        "recall": float(recall_50),
        "f1": float(f1_50),

        "ap50": float(ap50),
        "ap75": float(ap75),
        "map_50_95": float(map_50_95),

        "ap_by_iou": ap_by_iou,
        "precision_by_iou": precision_by_iou,
        "recall_by_iou": recall_by_iou,
        "f1_by_iou": f1_by_iou
    }


# MAIN
def main():
    print("CUDA available:", torch.cuda.is_available())
    print("Device:", device)

    print("\nLoading hard image names from CSV files...")

    hard_day = load_hard_image_names_from_csv(HARD_CSVS["day"])
    hard_night = load_hard_image_names_from_csv(HARD_CSVS["night"])
    hard_dawn = load_hard_image_names_from_csv(HARD_CSVS["dawn_dusk"])

    print("Hard day images:", len(hard_day))
    print("Hard night images:", len(hard_night))
    print("Hard dawn_dusk images:", len(hard_dawn))

    day_dataset = BDDHardTrainDataset(
        images_dir=BDD_IMAGES_DIR,
        ann_json_path=BDD_ANN_JSON,
        image_names=hard_day,
        split_label=0
    )

    night_dataset = BDDHardTrainDataset(
        images_dir=BDD_IMAGES_DIR,
        ann_json_path=BDD_ANN_JSON,
        image_names=hard_night,
        split_label=1
    )

    dawn_dataset = BDDHardTrainDataset(
        images_dir=BDD_IMAGES_DIR,
        ann_json_path=BDD_ANN_JSON,
        image_names=hard_dawn,
        split_label=2
    )

    all_hard_dataset = CombinedHardDataset([day_dataset, night_dataset, dawn_dataset])

    day_loader = DataLoader(
        day_dataset,
        batch_size=BATCH_SIZE_EXPERT,
        shuffle=True,
        collate_fn=collate_fn_list,
        num_workers=0
    )

    night_loader = DataLoader(
        night_dataset,
        batch_size=BATCH_SIZE_EXPERT,
        shuffle=True,
        collate_fn=collate_fn_list,
        num_workers=0
    )

    dawn_loader = DataLoader(
        dawn_dataset,
        batch_size=BATCH_SIZE_EXPERT,
        shuffle=True,
        collate_fn=collate_fn_list,
        num_workers=0
    )

    all_hard_loader_expert = DataLoader(
        all_hard_dataset,
        batch_size=BATCH_SIZE_EXPERT,
        shuffle=True,
        collate_fn=collate_fn_list,
        num_workers=0
    )

    all_hard_loader_gating = DataLoader(
        all_hard_dataset,
        batch_size=BATCH_SIZE_GATING,
        shuffle=True,
        collate_fn=collate_fn_list,
        num_workers=0
    )

    print("\nCreating 4 COCO-pretrained Faster R-CNN experts...")
    experts = [
        get_coco_faster_rcnn_trainable(train_backbone=TRAIN_BACKBONE).to(device)
        for _ in range(NUM_EXPERTS)
    ]

    print("\nFine-tuning experts on hard images...")

    experts[0] = train_one_expert(
        expert=experts[0],
        loader=day_loader,
        expert_name="Expert 0 - DAY hard images",
        num_epochs=NUM_EPOCHS_EXPERT,
        lr=LR_EXPERT
    )

    experts[1] = train_one_expert(
        expert=experts[1],
        loader=night_loader,
        expert_name="Expert 1 - NIGHT hard images",
        num_epochs=NUM_EPOCHS_EXPERT,
        lr=LR_EXPERT
    )

    experts[2] = train_one_expert(
        expert=experts[2],
        loader=dawn_loader,
        expert_name="Expert 2 - DAWN_DUSK hard images",
        num_epochs=NUM_EPOCHS_EXPERT,
        lr=LR_EXPERT
    )

    experts[3] = train_one_expert(
        expert=experts[3],
        loader=all_hard_loader_expert,
        expert_name="Expert 3 - ALL hard images",
        num_epochs=NUM_EPOCHS_EXPERT,
        lr=LR_EXPERT
    )

    print("\nSaving fine-tuned experts...")
    torch.save(
        {
            "expert_0_day": experts[0].state_dict(),
            "expert_1_night": experts[1].state_dict(),
            "expert_2_dawn_dusk": experts[2].state_dict(),
            "expert_3_all": experts[3].state_dict(),
        },
        EXPERT_SAVE_PATH
    )

    print("Saved experts to:", EXPERT_SAVE_PATH)

    print("\nCreating and training gating network...")
    gating_net = ImageGatingNetwork(num_experts=NUM_SPECIALISTS).to(device)

    gating_net = train_gating_network(
        gating_net=gating_net,
        loader=all_hard_loader_gating,
        num_epochs=NUM_EPOCHS_GATING,
        lr=LR_GATING
    )

    torch.save(gating_net.state_dict(), GATING_SAVE_PATH)
    print("Saved gating to:", GATING_SAVE_PATH)

    print("\nRunning final inference/evaluation on full BDD splits...")

    all_results = {}

    for split_name, split_txt_path in BDD_SPLITS.items():
        print("\n" + "=" * 80)
        print(f"Evaluating full BDD split: {split_name}")
        print("=" * 80)

        image_names = load_split_image_names(split_txt_path)

        dataset = BDDPersonDataset(
            images_dir=BDD_IMAGES_DIR,
            ann_json_path=BDD_ANN_JSON,
            image_names=image_names,
            split_name=split_name
        )

        loader = DataLoader(
            dataset,
            batch_size=1,
            shuffle=False,
            collate_fn=collate_fn_list,
            num_workers=0
        )

        metrics = evaluate_bdd_person(
            gating_net=gating_net,
            experts=experts,
            data_loader=loader,
            split_name=split_name
        )

        all_results[split_name] = metrics

        print("\n" + "-" * 75)
        print(f"BDD100K PERSON-ONLY RESULT AFTER HARD FINE-TUNING: {split_name}")
        print("-" * 75)
        print(f"Number of images          : {metrics['num_images']}")
        print(f"GT person boxes           : {metrics['num_gt']}")
        print(f"Predicted person boxes    : {metrics['num_pred']}")
        print(f"Precision @ IoU=0.50      : {metrics['precision']:.4f}")
        print(f"Recall @ IoU=0.50         : {metrics['recall']:.4f}")
        print(f"F1-score @ IoU=0.50       : {metrics['f1']:.4f}")
        print(f"AP50 / mAP@0.5 person     : {metrics['ap50']:.4f}")
        print(f"AP75 / mAP@0.75 person    : {metrics['ap75']:.4f}")
        print(f"mAP@0.5:0.95 person       : {metrics['map_50_95']:.4f}")

        print("\nAP by IoU threshold:")
        for iou_t, ap in metrics["ap_by_iou"].items():
            print(f"AP@{iou_t:.2f}: {ap:.4f}")

        print("-" * 75)

    print("\n" + "=" * 115)
    print("FINAL SUMMARY - HARD FINE-TUNED MoE ON BDD100K PERSON ONLY")
    print("=" * 115)

    print(
        f"{'Split':12s} | "
        f"{'Images':>8s} | "
        f"{'GT':>8s} | "
        f"{'Pred':>8s} | "
        f"{'AP50':>8s} | "
        f"{'AP75':>8s} | "
        f"{'mAP50-95':>10s} | "
        f"{'Prec50':>8s} | "
        f"{'Recall50':>8s} | "
        f"{'F1-50':>8s}"
    )

    print("-" * 115)

    for split_name, m in all_results.items():
        print(
            f"{split_name:12s} | "
            f"{m['num_images']:8d} | "
            f"{m['num_gt']:8d} | "
            f"{m['num_pred']:8d} | "
            f"{m['ap50']:8.4f} | "
            f"{m['ap75']:8.4f} | "
            f"{m['map_50_95']:10.4f} | "
            f"{m['precision']:8.4f} | "
            f"{m['recall']:8.4f} | "
            f"{m['f1']:8.4f}"
        )

    print("=" * 115)

    return all_results


if __name__ == "__main__":
    all_results = main()